# H2: has the "innate ability" of champion squads increased?

> An analyst claims that today, thanks to progress and improved conditions, the use and
> flourishing of players' innate ability has improved compared to the past — for example,
> the average innate ability of the champion team's players over the last 2 seasons has
> been greater than over the 2 seasons before that. He defines innate ability as the ratio
> of a player's experience to their age. Examine this claim for the stated example, and for
> the active players of that team in that season.

The analyst hands over his own metric, so the arithmetic is settled before we start. What is
not settled is what that metric measures, and that turns out to be most of the answer.

## Reading the question

"The last two seasons" are 2024-25 and 2025-26, won by Oklahoma City and New York. "The two
seasons before that" are 2022-23 and 2023-24, won by Denver and Boston. The most recent
scrape brought 2025-26 in as a finished season, so this window sits a year later than the one
the original bootcamp analysis used.

"Active players of that team in that season" means on the champion's roster and on court for
at least one game. That gives 14 players for Denver, 15 for Boston, 18 for Oklahoma City and
17 for New York: 35 in the recent pair against 29 in the earlier one.

"Innate ability" is the analyst's definition, seasons played before this one divided by age
during this one. Both halves have to be the right vintage. Age has to be his age in that
season rather than his age now, or every season would carry the same number and the
comparison would have nothing left to measure. Experience has to be seasons played before the
season in question, so a rookie scores 0.

Experience is taken from the roster page, which prints the figure per season instead of
reconstructing it. The D2 notebook compared that source against the derived alternative on
the same kind of population and found the derived one wrong where a player had missed a full
season; the check below repeats it in one line for this population.

**Pooled for the test, split for the reading.** The claim is about two windows, so the test
compares two windows. But 35 players drawn from two different clubs are not one population,
and the four squads differ enough that a pooled average describes none of them. Each squad is
shown on its own alongside the pooled test.

## The hypothesis, before anything is computed

- **H₀**: active players on the champion squad have the same average innate ability in the
  recent pair of seasons as in the earlier pair.
- **H₁**: the two differ.
- **α = 0.05.**
- **Two-tailed.**

That last line is the decision this notebook exists to get right. The analyst asserts an
increase, which invites a one-tailed test pointed that way. A one-tailed test in the
"greater" direction can return two things: evidence of an increase, or nothing. It has no way
of reporting a decrease, and when it meets one it returns a p-value close to 1, which reads
like strong agreement with H₀ if the number is all you look at.

That is what happened the first time this question was answered. The original ran
`alternative='greater'`, reported p ≈ 0.97 and "cannot reject H₀", and printed a t-statistic
of −4.918 beside it. The minus sign was the part that mattered: the sample had moved hard in
the direction the test was blind to, and "we cannot conclude it went up" got written down in
a way that hid "it went down".

A change in either direction answers the analyst's question, so the test here is two-tailed.
The one-tailed version is computed as well, further down, because the gap between the two is
the most useful thing in this notebook.

The original also passed each group through its own Yeo-Johnson transform before testing.
Fitting a separate transform per group sends the two groups through two different functions,
after which any difference in location is partly a difference in the transform. Nothing here
is transformed.

## What the data has to supply

Four lists of people, one per season: everyone signed to that season's championship-winning
club who got onto the court at least once.

For each of them, two numbers as they stood in that season. How old he was, and how many
seasons of professional basketball he had behind him when it started. Both change every year,
so both have to be read off that season's record rather than off his profile today.

Age is printed on the season's own statistics page. Experience is printed next to each name
on the club's roster page for that season, and that is the version used here. The alternative
is to work it backwards from a career total, which breaks for anyone who missed a full year.

In [1]:
import _setup  # noqa: F401

import pandas as pd

import utils.custom_plots as cp
import utils.custom_stats as cs
from utils.db_utils import run_query

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# The four champion seasons this question is asked over.
EARLIER_SEASONS = (2023, 2024)  # 2022-23 Denver, 2023-24 Boston
RECENT_SEASONS = (2025, 2026)  # 2024-25 Oklahoma City, 2025-26 New York

RECENT_LABEL = "recent (2024-25, 2025-26)"
EARLIER_LABEL = "earlier (2022-23, 2023-24)"

ALPHA = 0.05

In [ ]:
SQL_CHAMPIONS = """
with roster_experience as (
    -- The experience figure the roster page states outright, one row per
    -- player-season. A player traded mid-season has a roster row at each club
    -- and both carry the same figure, so max() only collapses the duplicate.
    select season,
           player_id,
           max(experience_seasons) as experience_roster
    from processed.rosters
    where season between :first_season and :last_season
    group by season, player_id
)
select ps.season,
       ps.season_label,
       ds.champion_team_name,
       ps.player_id,
       ps.player_name,
       ps.position,
       ps.games_played,
       ps.age,
       re.experience_roster,
       ps.experience_seasons as experience_rolled_back
from analyst_ready.player_season ps
join analyst_ready.dim_season ds
  on ds.season = ps.season
left join roster_experience re
  on re.season = ps.season
 and re.player_id = ps.player_id
where ps.season between :first_season and :last_season
  and ps.is_on_champion_team
  and ps.games_played > 0
order by ps.season, ps.player_name
"""

raw = run_query(
    SQL_CHAMPIONS,
    {"first_season": EARLIER_SEASONS[0], "last_season": RECENT_SEASONS[1]},
)

# SQL numeric arrives as Decimal typed object; custom_plots skips those silently.
for col in ("age", "experience_roster", "experience_rolled_back"):
    raw[col] = raw[col].astype(float)

print(raw.shape)
print(raw.groupby(["season_label", "champion_team_name"]).size())
raw.head()

(64, 10)
season_label  champion_team_name   
2022-23       Denver Nuggets           14
2023-24       Boston Celtics           15
2024-25       Oklahoma City Thunder    18
2025-26       New York Knicks          17
dtype: int64


,season,season_label,champion_team_name,player_id,player_name,position,games_played,age,experience_roster,experience_rolled_back
0,2023,2022-23,Denver Nuggets,gordoaa01,Aaron Gordon,PF,68,27.0,8.0,8.0
1,2023,2022-23,Denver Nuggets,brownbr01,Bruce Brown,SF,80,26.0,4.0,4.0
2,2023,2022-23,Denver Nuggets,braunch01,Christian Braun,SG,76,21.0,0.0,0.0
3,2023,2022-23,Denver Nuggets,jordade01,DeAndre Jordan,C,39,34.0,14.0,14.0
4,2023,2022-23,Denver Nuggets,smithis01,Ish Smith,PG,43,34.0,12.0,NaN


In [3]:
analysis = raw.copy()

# The analyst's metric, exactly as he defines it.
analysis["experience_seasons"] = analysis["experience_roster"]
analysis["innate_ability"] = analysis["experience_seasons"] / analysis["age"]

# Recent is listed first, so it is group A in every comparison below and a
# positive estimate reads as "recent above earlier".
analysis["window"] = pd.Categorical(
    analysis["season"].map(
        lambda s: RECENT_LABEL if s in RECENT_SEASONS else EARLIER_LABEL
    ),
    categories=[RECENT_LABEL, EARLIER_LABEL],
    ordered=True,
)
analysis["squad"] = analysis["season_label"] + " · " + analysis["champion_team_name"]

# 1. Does the stated experience figure cover everyone in the population?
disagree = (
    analysis["experience_rolled_back"].notna()
    & analysis["experience_roster"].ne(analysis["experience_rolled_back"])
).sum()
print(f"players: {len(analysis)}")
print(f"  without a stated (roster) figure : {analysis['experience_roster'].isna().sum()}")
print(f"  without a derived figure         : {analysis['experience_rolled_back'].isna().sum()}")
print(f"  the two disagree                 : {disagree}")

# 2. Is age the age during that season, or one current age repeated?
repeats = (
    analysis.pivot_table(index="player_name", columns="season_label", values="age")
    .dropna(thresh=2)
)
print(f"\nplayers appearing in more than one of these four seasons: {len(repeats)}")
print(repeats.head())

# 3. How heavily tied is the ratio?
print(
    f"\ninnate_ability: {analysis['innate_ability'].nunique()} distinct values across "
    f"{len(analysis)} players, {(analysis['innate_ability'] == 0).sum()} of them exactly 0"
)
print(analysis.groupby("window", observed=True).size())

players: 64
  without a stated (roster) figure : 0
  without a derived figure         : 5
  the two disagree                 : 1

players appearing in more than one of these four seasons: 1
season_label  2022-23  2023-24  2024-25  2025-26
player_name                                     
Dillon Jones      NaN      NaN     23.0     24.0

innate_ability: 37 distinct values across 64 players, 11 of them exactly 0
window
recent (2024-25, 2025-26)     35
earlier (2022-23, 2023-24)    29
dtype: int64


### What the checks found

Nothing needed patching. All 64 active champion players have a stated experience figure from
the roster page. The derived alternative is missing for 5 of them and disagrees with the
stated figure for a sixth, which is why the stated one is used.

Only one player appears on two of these four squads, since no club repeated as champion, and
he is listed a year older the second time. Age moves with the season rather than sitting at
one current value, which is the property this comparison depends on.

The ratio takes 37 distinct values across 64 players, and 11 of them are exactly 0: a
rookie's experience is 0, so his innate ability is 0 whatever his age. That matters later,
because a variable this heavily tied resists a normal approximation and can leave a bootstrap
interval coarse.

## The four squads, described

The metric first, then its two components, for each champion squad on its own. At 14 to 18
players a group the summary table sits close to the raw data anyway, so the spread is worth
as much attention as the average.

In [4]:
METRICS = ["innate_ability", "age", "experience_seasons"]

by_squad = pd.concat(
    [
        cs.summary_stats(sub[METRICS]).assign(squad=name)
        for name, sub in analysis.groupby("squad", sort=True)
    ],
    ignore_index=True,
)
by_squad[
    ["squad", "column", "n", "mean", "median", "std", "min", "q1", "q3", "max"]
].round(3)

,squad,column,n,mean,median,std,min,q1,q3,max
0,2022-23 · Denver Nuggets,innate_ability,14,0.194,0.177,0.145,0.0,0.098,0.307,0.412
1,2022-23 · Denver Nuggets,age,14,26.786,25.500,4.917,20.0,24.250,28.500,36.000
2,2022-23 · Denver Nuggets,experience_seasons,14,5.786,4.500,4.995,0.0,2.250,8.750,14.000
3,2023-24 · Boston Celtics,innate_ability,15,0.180,0.192,0.133,0.0,0.080,0.245,0.432
4,2023-24 · Boston Celtics,age,15,26.533,26.000,4.373,19.0,24.500,28.000,37.000
5,2023-24 · Boston Celtics,experience_seasons,15,5.267,5.000,4.636,0.0,2.000,6.500,16.000
6,2024-25 · Oklahoma City Thunder,innate_ability,18,0.099,0.089,0.089,0.0,0.010,0.190,0.233
7,2024-25 · Oklahoma City Thunder,age,18,24.500,25.000,2.618,21.0,22.250,25.750,30.000
8,2024-25 · Oklahoma City Thunder,experience_seasons,18,2.556,2.000,2.479,0.0,0.250,4.750,7.000
9,2025-26 · New York Knicks,innate_ability,17,0.162,0.160,0.118,0.0,0.043,0.259,0.333


Oklahoma City is the outlier here, not the recent window. Denver averaged 0.194 on the ratio,
Boston 0.180, New York 0.162, and Oklahoma City 0.099, roughly half of any of the other
three. Its median is 0.09 against 0.16 to 0.19 elsewhere.

Pooled, the recent pair averages 0.129 against 0.187 for the earlier pair, a gap of 0.057
pointing the wrong way for the analyst's claim. Almost all of it comes from that one squad.
Set Oklahoma City aside and New York's 0.162 sits inside the range Denver and Boston made.

Both components moved together. Oklahoma City's squad averaged 24.5 years old and 2.6 prior
seasons; Denver's averaged 26.8 and 5.8. New York sits between them at 26.1 and 4.6, closer
to the earlier pair than to its own window-mate.

Spread is wide everywhere. Every squad runs from 0 up to somewhere between 0.23 and 0.43, and
the standard deviations are 0.09 to 0.15 on means of 0.10 to 0.19. At 14 to 18 players a
side, none of these are tightly pinned numbers.

In [5]:
fig_squads = cp.grouped_box_plot(
    analysis,
    group_col="squad",
    value_col="innate_ability",
    show_points="all",
    sort_by="median",
    orientation="horizontal",
    title="Innate ability (experience / age) by champion squad",
)
fig_squads

Every group carries the gold low-n outline. A champion roster is 14 to 18 people, so there is
no version of this question with 30 players in a group. The flag describes the question
rather than a fault in the data, and it is the reason the intervals below matter more than
the p-values.

Oklahoma City's box is the one that moves. Its upper hinge sits at 0.19, roughly where the
other three put their medians, and its whisker stops at 0.23. The most experienced player on
that title-winning squad was Alex Caruso, 7 seasons and 30 years old. Denver and Boston each
carried a fourteen-to-sixteen-season veteran into his thirties, which is where their long
right tails come from.

New York's box overlaps Denver's and Boston's across most of its width. Of the two squads in
the recent window, one looks like the earlier window and one does not.

In [6]:
fig_ecdf = cp.ecdf_plot(
    analysis,
    cols="innate_ability",
    group_col="window",
    mark_percentiles=[0.25, 0.5, 0.75],
    title="Innate ability, cumulative: share of each window at or below x",
)
fig_ecdf

The cumulative view drops the mean entirely. Each curve reads as "what share of this window
sat at or below this value". The recent curve runs above the earlier one across almost the
whole axis, so a larger share of recent players sits below any threshold you pick. The shift
is consistent rather than concentrated at one point.

The two curves start together. 17% of each window had never played a season before, so the
left edge is not what separates them. They separate through the middle, where the recent
median is 0.091 and the earlier one 0.192.

They close again at the top. Both windows carry a few thirty-somethings, and the recent
ceiling of 0.33 (Jordan Clarkson, 33 years old with 11 seasons behind him) is lower than the
earlier ceiling of 0.43 (Al Horford at 37 with 16), but not by much.

## How much of this survives the sample size

Two windows of 35 and 29 players, built from four squads of 14 to 18. At that size a group
mean is not pinned down, so it needs an interval around it rather than a third decimal place.
The interval around the difference then carries more information than the verdict attached to
it.

In [7]:
window_means = cs.bootstrap_ci(analysis, value_col="innate_ability", group_col="window")
squad_means = cs.bootstrap_ci(analysis, value_col="innate_ability", group_col="squad")

pd.concat([window_means, squad_means], ignore_index=True)[
    ["group", "n", "estimate", "ci_low", "ci_high", "se", "method", "flags"]
].round(3)

,group,n,estimate,ci_low,ci_high,se,method,flags
0,"recent (2024-25, 2025-26)",35,0.129,0.095,0.167,0.018,BCa,
1,"earlier (2022-23, 2023-24)",29,0.187,0.140,0.237,0.025,BCa,
2,2022-23 · Denver Nuggets,14,0.194,0.121,0.266,0.037,BCa,small n (14): interval is optimistic
3,2023-24 · Boston Celtics,15,0.180,0.121,0.253,0.033,BCa,small n (15): interval is optimistic
4,2024-25 · Oklahoma City Thunder,18,0.099,0.062,0.141,0.020,BCa,small n (18): interval is optimistic
5,2025-26 · New York Knicks,17,0.162,0.108,0.216,0.028,BCa,small n (17): interval is optimistic


The two window intervals overlap, narrowly. The recent mean lands between 0.095 and 0.167,
the earlier one between 0.140 and 0.237, leaving a shared strip from 0.140 to 0.167. Neither
interval excludes the other's point estimate, which is already a hint at how the test will
come out.

Every per-squad row carries the same warning, and it is the right one. At n between 14 and 18
a bootstrap interval is optimistic, because the resamples can only ever contain values
already observed. Nothing worse showed up: the ratio is heavily tied, which can leave a BCa
interval undefined, and all six came back finite.

Oklahoma City's interval, 0.062 to 0.141, is the lowest of the four, and its top edge sits
below the point estimate of every other squad.

## The test

Two-tailed, α = 0.05, as fixed above. `test='auto'` screens normality and equal variance
itself and records what it decided in `chosen_because`, which is worth reading before the
p-value.

In [8]:
REPORT_COLS = [
    "comparison", "group_sizes", "test", "chosen_because", "estimate_type", "estimate",
    "ci_low", "ci_high", "p_value", "decision", "effect_size", "effect_type",
    "magnitude", "alternative", "flags",
]

# The screen `auto` runs internally, printed so its choice can be checked.
print(
    pd.concat(
        [
            cs.normality_test(sub[["innate_ability"]]).assign(window=name)
            for name, sub in analysis.groupby("window", observed=True)
        ],
        ignore_index=True,
    )[["window", "n", "method", "statistic", "p_value", "decision", "skew"]].round(4)
)
print()
print(cs.variance_test(analysis, group_col="window", value_col="innate_ability")[
    ["test", "group_sizes", "p_value", "decision", "max_var_ratio"]
].round(4))

primary = cs.compare_groups(
    analysis,
    group_col="window",
    value_col="innate_ability",
    test="auto",
    alpha=ALPHA,
    alternative="two-sided",
    ci="bootstrap",
)
primary[REPORT_COLS].round(4).T

                       window   n   method  statistic  p_value           decision    skew
0   recent (2024-25, 2025-26)  35  shapiro     0.8951   0.0029          reject H₀  0.3929
1  earlier (2022-23, 2023-24)  29  shapiro     0.9426   0.1173  fail to reject H₀  0.2640

             test                                        group_sizes  p_value           decision  max_var_ratio
0  levene(median)  recent (2024-25, 2025-26)=35; earlier (2022-23...   0.2486  fail to reject H₀         1.6084


,0
comparison,"recent (2024-25, 2025-26) vs earlier (2022-23,..."
group_sizes,"recent (2024-25, 2025-26)=35; earlier (2022-23..."
test,mannwhitney
chosen_because,"auto: normal 1/2, variances equal"
estimate_type,median difference
estimate,-0.1014
ci_low,-0.19
ci_high,0.0077
p_value,0.1075
decision,fail to reject H₀


The screen first. The recent window fails Shapiro-Wilk at p = 0.003, driven by the six
players sitting at exactly 0, while the earlier window passes at p = 0.117. Levene finds the
spreads comparable, with a variance ratio of 1.6. `chosen_because` records that as
`normal 1/2, variances equal`, and one non-normal group is enough to send the comparison to
Mann-Whitney. So the estimate is a median difference and the effect size is Cliff's delta.

Recent is group A, so a negative estimate means the recent window sits below the earlier one.
It does. The median difference is −0.101, with a bootstrap interval from −0.190 to +0.008.

**p = 0.108, two-tailed. H₀ is not rejected at α = 0.05.** Cliff's delta is −0.235, which the
toolkit calls small. Delta is a statement about pairs: draw one player from each window and
the earlier one has the higher ratio 60% of the time against 36% the other way, with the
remaining 4% tied.

The interval is the more useful line. It runs from a median 0.19 lower in the recent window
to 0.01 higher, so most of it sits below zero without excluding zero. Read together with the
effect size, that says the recent squads look less experienced for their age and the sample
is too small to make the difference stick.

## The same data, read three other ways

The first row below is the test the original ran: the same comparison, pointed one way. The
other two drop the rank test for a mean-based one and for a permutation test, to check that
the verdict does not rest on which test `auto` happened to pick.

In [9]:
alternatives = pd.concat(
    [
        cs.compare_groups(
            analysis, group_col="window", value_col="innate_ability",
            test="auto", alpha=ALPHA, alternative="greater",
        ).assign(reading="one-tailed 'greater' (the original's test)"),
        cs.compare_groups(
            analysis, group_col="window", value_col="innate_ability",
            test="welch", alpha=ALPHA,
        ).assign(reading="Welch on the untransformed means"),
        cs.compare_groups(
            analysis, group_col="window", value_col="innate_ability",
            test="permutation", alpha=ALPHA,
        ).assign(reading="permutation on the difference in means"),
    ],
    ignore_index=True,
)
alternatives[
    ["reading", "test", "alternative", "statistic", "estimate_type", "estimate",
     "p_value", "decision", "effect_size", "magnitude"]
].round(4)

,reading,test,alternative,statistic,estimate_type,estimate,p_value,decision,effect_size,magnitude
0,one-tailed 'greater' (the original's test),mannwhitney,greater,388.0000,median difference,-0.1014,0.9477,fail to reject H₀,-0.2355,small
1,Welch on the untransformed means,welch,two-sided,-1.8347,mean difference,-0.0573,0.0722,fail to reject H₀,-0.4653,small
2,permutation on the difference in means,permutation,two-sided,-0.0573,mean difference,-0.0573,0.0656,fail to reject H₀,-0.4653,small


**The one-tailed test the original ran returns p = 0.948.** Nothing about that contradicts
the two-sided 0.108. Same 64 players, same U statistic of 388, same median difference of
−0.101. Only the question changed. For a "greater" test on a sample that moved the other way,
the one-tailed p is about `1 − (two-sided p) / 2`, which here is 0.946 against the 0.948 the
rank test produced. So the 0.948 carries no information the 0.108 does not. It restates "the
sample went down" in a form that looks like agreement with H₀.

**A one-tailed p above 0.5 is never evidence for H₀.** It means the data moved against the
alternative, which the test cannot report because it was not asked to. Reading 0.95 as
"strongly consistent with no change" gets the finding backwards. On the original's window
that was expensive, because the number sat next to a t-statistic of −4.918: a decrease large
enough that a two-sided test would have flagged it at once. On this window the decrease is
smaller and the two-sided test does not reach α either, so the verdict happens to match. That
the verdict matches is luck rather than method.

**The result does not depend on which test is used.** Welch on the untransformed means gives
t = −1.83, p = 0.072, Hedges' g = −0.47. A permutation test on the same difference gives
p = 0.066 without assuming any distribution at all. Both point the same way as the rank test
and neither clears 0.05. Whatever the original's per-group Yeo-Johnson was buying, it was not
needed here.

## What the ratio actually measures

A ratio hides which of its two parts moved. Here both did, so the components are worth
testing on their own, and the relationship between them is worth looking at directly.

In [10]:
# Age minus experience is roughly the age at which he entered the league.
analysis["entry_age"] = analysis["age"] - analysis["experience_seasons"]

components = pd.concat(
    [
        cs.compare_groups(
            analysis, group_col="window", value_col=col,
            test="auto", alpha=ALPHA, ci="bootstrap",
        ).assign(component=col)
        for col in ("age", "experience_seasons", "entry_age")
    ],
    ignore_index=True,
)
print(analysis.groupby("window", observed=True)[
    ["age", "experience_seasons", "entry_age", "innate_ability"]
].mean().round(2))

components[
    ["component", "test", "estimate_type", "estimate", "ci_low", "ci_high",
     "p_value", "decision", "effect_size", "magnitude", "flags"]
].round(4)

                              age  experience_seasons  entry_age  innate_ability
window                                                                          
recent (2024-25, 2025-26)   25.29                3.54      21.74            0.13
earlier (2022-23, 2023-24)  26.66                5.52      21.14            0.19


,component,test,estimate_type,estimate,ci_low,ci_high,p_value,decision,effect_size,magnitude,flags
0,age,welch,mean difference,-1.3695,-3.395,0.5409,0.1787,fail to reject H₀,-0.3497,small,
1,experience_seasons,mannwhitney,median difference,-3.0000,-6.000,0.0000,0.1318,fail to reject H₀,-0.2197,small,
2,entry_age,mannwhitney,median difference,1.0000,NaN,NaN,0.1493,fail to reject H₀,0.2079,small,bootstrap interval undefined: the statistic ba...


Neither component clears α on its own. Age is 1.37 years lower in the recent window (Welch,
p = 0.179, interval −3.4 to +0.5 years). Median experience is 3 seasons lower (Mann-Whitney,
p = 0.132, interval −6 to 0). The ratio combines two differences that are individually too
small to call, which is why it lands where it does.

Entry age went the other way, up by a median of 1 season. That row also carries the flag
worth reading: the bootstrap interval came back undefined, because a median difference of one
whole season barely moves across resamples when the underlying values are small integers. The
toolkit suggests `method='percentile'` for that case; the honest reading is that a one-season
median difference on 64 players is not something an interval can resolve.

Put together, both halves of the ratio pushed it down. Recent champion players are on average
1.4 years younger and entered the league about half a year later, so a smaller fraction of
their lives has been spent in the NBA.

In [11]:
print(
    cs.correlation_test(analysis[["age", "experience_seasons"]])[
        ["x", "y", "n", "r", "ci_low", "ci_high", "r_squared", "p_value", "magnitude"]
    ].round(3)
)

fig_components = cp.scatter_plot(
    analysis,
    x="age",
    y="experience_seasons",
    color_by="window",
    trendline=True,
    title="What the ratio is made of: age against seasons already played",
)
fig_components

     x                   y   n      r  ci_low  ci_high  r_squared  p_value magnitude
0  age  experience_seasons  64  0.914   0.862    0.947      0.836      0.0     large


Age and experience correlate at r = 0.91 across the 64 players, and the fit inside each
window is tight on its own (r² = 0.89 for the earlier pair, 0.74 for the recent one). Knowing
a champion player's age tells you most of his experience. That is the point of this figure,
and it is worth saying what it means for the metric.

Rearrange the ratio. A player's experience is his age minus the age he entered the league, so
`experience / age` equals `1 − entry_age / age`. Nothing in that is innate. It says how much
of a player's life has been spent in the NBA, which is career stage expressed as a fraction.

Two things follow. The ratio rises mechanically every year a player stays in the league, so a
squad scores high by being old and low by being young. And a rookie scores exactly 0 however
good he is. Five of Oklahoma City's eighteen players were rookies, and all five put a zero
into that squad's average.

## Conclusion

**The claim is not supported. Innate ability as the analyst defines it went down rather than
up, and the decrease is not large enough to call reliable.**

Active players on the last two champion squads averaged 0.129 on `experience / age`, against
0.187 for the two squads before them. The median difference is −0.101, two-tailed p = 0.108
on a Mann-Whitney test, Cliff's delta −0.235. H₀ is not rejected at α = 0.05, so the honest
statement is that this data cannot separate the two windows. What it certainly does not do is
support an increase. Every reading points the other way, and a mean test and a permutation
test agree at p = 0.072 and p = 0.066.

**On the test the original ran.** Its one-tailed version returns p = 0.948 here, which is
evidence of nothing beyond the sample having moved in the direction the test was blind to.
Printed without the sign of the statistic beside it, that number looks like agreement with
H₀ when it means the opposite. Running two-tailed costs nothing, and on the original's window
it would have caught a much larger decrease immediately.

**On the metric.** `experience / age` measures career stage, not innate ability. It is
`1 − entry_age / age`, it rises with every season a player survives, and it hands a rookie
a 0. The difference measured here is a fact about roster construction. Oklahoma City won the
2024-25 title with a squad averaging 24.5 years old and 2.6 seasons of experience, five of
them rookies, and that squad alone is most of the gap. New York's 2025-26 squad, at 0.162,
sits inside the range Denver and Boston made. Nobody's ability failed to flourish.

**Limits.** Four squads, 64 players, 14 to 18 a side. Each squad is one front office's
decision about how to build one team, and one of the four is unusual. A window covering ten
champions would show whether young title squads are a trend; four cannot. And the metric came
with the question. A different definition of innate ability would produce a different answer,
which is the strongest argument for reading the two components rather than the ratio.